In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy.signal import savgol_filter
from mpl_toolkits.mplot3d import Axes3D
import adaptive_latents.stimulation.photostimulation as ps
from adaptive_latents.transforms.proSVD import proSVD
from scipy.signal import savgol_filter
from adaptive_latents import CONFIG

In [ ]:

# Specify the paths to the files
stim_file_path = '../workspace/datasets/fish/output_020424_ds1/stimmed.txt'
C_file_path = '../workspace/datasets/fish/output_020424_ds1/analysis_proc_C.txt'
photo_file_path = '../workspace/datasets/fish/output_020424_ds1/photostims.npy'

# Load the files
stim = np.loadtxt(stim_file_path) 
"""1st entry: frame number,
2nd entry: ignore,
3rd entry: angle of motion L,
4th entry: angle of motion R,
5th entry: timestamp,"""
C = np.loadtxt(C_file_path)#
"""Calcium imaging. 
1st entry is neuron ID,
2nd is time (frame)"""
photostim = np.load(photo_file_path)
"""1st entry: frame number,
2nd entry: counter of stims,
3rd entry: neuron ID,
4th entry: position X of neuron,
5th entry: position Y of neuron,"""

# extra note: Fs= 2.3 Hz

In [ ]:
C_visual= C[:, :1114]
C_photostim = C[:, 1114:]

In [ ]:
#Dimensionality reduction 
pro = proSVD(3, centering=True) # like the PCA() step
pro.run_on(C_visual) # like the fit_transform step
small_C=pro.project(C_photostim).T # like the transform step

In [ ]:
small_C.shape

## Animation

In [ ]:
# Smoothing function
def smooth_trajectory(data, window_length=24, polyorder=3):
    return savgol_filter(data, window_length, polyorder, axis=0)


In [ ]:
# Animate function
def animate_trajectories(data, start_frame=0, end_frame=None, output_file='trajectory_animation2.mp4'):
    if end_frame is None or end_frame > len(data):
        end_frame = len(data)
    
    # Smooth the data
    smoothed_data = smooth_trajectory(data[start_frame:end_frame])

    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    ax.set_xlim(np.min(smoothed_data[:, 0]), np.max(smoothed_data[:, 0]))
    ax.set_ylim(np.min(smoothed_data[:, 1]), np.max(smoothed_data[:, 1]))
    ax.set_zlim(np.min(smoothed_data[:, 2]), np.max(smoothed_data[:, 2]))

    # Initialize the line
    line, = ax.plot([], [], [], lw=2)

    def init():
        line.set_data([], [])
        line.set_3d_properties([])
        return line,

    def update(num):
        window_size = 205  # Length of the visible segment
        start_idx = max(0, num - window_size)
        end_idx = num + 1
        
        line.set_data(smoothed_data[start_idx:end_idx, 0], smoothed_data[start_idx:end_idx, 1])
        line.set_3d_properties(smoothed_data[start_idx:end_idx, 2])
        return line,

    ani = animation.FuncAnimation(fig, update, frames=len(smoothed_data), init_func=init, blit=True)

    # Save the animation as an mp4 file
    ani.save(output_file, writer='ffmpeg', fps=30)
    plt.close()



In [ ]:
output_file = 'trajectory2_animation.mp4'
animate_trajectories(small_C, start_frame=1, end_frame=1000)

## Now with colors per photostim

In [ ]:
def rank_neurons(stim, C, photostim):
    # Create DataFrame for stim
    stim_columns = ['frame_number', 'ignore', 'angle_of_motion_L', 'angle_of_motion_R', 'timestamp']
    stim_df = pd.DataFrame(stim, columns=stim_columns)

    # Create DataFrame for C, where each column after the first represents a neuron
    neuron_ids = [f'neuron_{i}' for i in range(C.shape[0])]
    time_points = [f'frame_{i}' for i in range(C.shape[1])]
    C_df = pd.DataFrame(C.T, columns=neuron_ids, index=time_points)

    # Create DataFrame for photostim
    photostim_columns = ['frame_number', 'stim_counter', 'neuron_ID', 'position_X', 'position_Y']
    photostim_df = pd.DataFrame(photostim, columns=photostim_columns)

    # Handle neurons with two trials in the data frame: 14 and 98 turn into 15 and 99 in the second trial
    neuron_position_counts = photostim_df.groupby('neuron_ID')['position_X'].count()

    # Filter neurons with more than 6 occurrences
    neurons_with_many_positions = neuron_position_counts[neuron_position_counts > 6].index

    # Filter the DataFrame to include only these neurons
    filtered_neurons_df = photostim_df[photostim_df['neuron_ID'].isin(neurons_with_many_positions)]

    # Function to modify the neuron ID for the second group of 5 occurrences
    def modify_neuron_ids(df):
        modified_ids = df.copy()
        neuron_groups = modified_ids.groupby('neuron_ID')
        
        for neuron_id, group in neuron_groups:
            # Split the group into sets of 5
            for i in range(1, len(group) // 5 + 1):
                start_idx = i * 5
                if start_idx < len(group):
                    modified_ids.loc[group.index[start_idx:start_idx + 5], 'neuron_ID'] = str(int(neuron_id) + 1)
        
        return modified_ids

    # Apply the function to modify the neuron IDs in the original DataFrame
    modified_photostim_df = modify_neuron_ids(photostim_df)
    # Drop the last row by index (we only have one stimulation for the last neuron)
    modified_photostim_df = modified_photostim_df.drop(modified_photostim_df.index[-1])

    # Ensure the frame_number and neuron_ID are of integer type
    modified_photostim_df['frame_number'] = modified_photostim_df['frame_number'].astype(int)
    modified_photostim_df['neuron_ID'] = modified_photostim_df['neuron_ID'].astype(int)

    # Sort the df by 'frame_number'
    modified_photostim_df = modified_photostim_df.sort_values(by='frame_number').reset_index(drop=True)

    return modified_photostim_df

In [ ]:
neurons_df=rank_neurons(stim, C, photostim) 
index_df=neurons_df.drop_duplicates(subset='neuron_ID', keep='first')
# Extract and adjust indices based on 'frame_number' from index_df
indices = index_df['frame_number'].values - 1114

# Convert indices to integers
indices = indices.astype(int)

In [ ]:
# Animate function
def animate_trajectories(data, highlight_indices, start_frame=0, end_frame=None, output_file='trajectory_animation.mp4'):
    if end_frame is None or end_frame > len(data):
        end_frame = len(data)

    # Smooth the data
    smoothed_data = smooth_trajectory(data[start_frame:end_frame])

    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    ax.set_xlim(np.min(smoothed_data[:, 0]), np.max(smoothed_data[:, 0]))
    ax.set_ylim(np.min(smoothed_data[:, 1]), np.max(smoothed_data[:, 1]))
    ax.set_zlim(np.min(smoothed_data[:, 2]), np.max(smoothed_data[:, 2]))

    # Parameters for animation
    sweep_duration = 25
    hold_duration = 15
    total_duration = sweep_duration + hold_duration
    fps = 30

    # Initialize the line
    line, = ax.plot([], [], [], lw=2, color='blue')

    # Setup the writer
    writer = animation.FFMpegWriter(fps=fps, bitrate=500)
    
    with writer.saving(fig, output_file, 100):
        pON = False
        count = 0
        pcount = -1
        for i in range(start_frame, start_frame + 600):
            ax.cla()  # Clear the axes
            ax.set_xlim(np.min(smoothed_data[:, 0]), np.max(smoothed_data[:, 0]))
            ax.set_ylim(np.min(smoothed_data[:, 1]), np.max(smoothed_data[:, 1]))
            ax.set_zlim(np.min(smoothed_data[:, 2]), np.max(smoothed_data[:, 2]))
            
            # Add axis labels
            ax.set_xlabel('X axis')
            ax.set_ylabel('Y axis')
            ax.set_zlabel('Z axis')

            ax.plot(smoothed_data[start_frame:, 0], smoothed_data[start_frame:, 1], smoothed_data[start_frame:, 2], lw=1, color='blue', alpha=0.1, zorder=5)
            ax.plot(smoothed_data[i-20:i, 0], smoothed_data[i-20:i, 1], smoothed_data[i-20:i, 2], lw=2, color='orange', zorder=10)
            
            if (i in highlight_indices) or pON:
                ax.plot(smoothed_data[i-count:i, 0], smoothed_data[i-count:i, 1], smoothed_data[i-count:i, 2], lw=3, color='red', zorder=10)
                pON = True
                count += 1
                if count > 6:
                    count = 0
                    pON = False
                    pcount += 1

            writer.grab_frame()

In [ ]:
output_file = 'trajectory_animation3.mp4'
animate_trajectories(small_C, indices ,start_frame=0, end_frame=1000, output_file=output_file)

Indices for intermidiate photostims:

In [ ]:
# Identify the index of the first occurrence of each unique value in neuron_ID
first_occurrences = neurons_df.drop_duplicates(subset=['neuron_ID'], keep='first').index

# Drop these rows from the DataFrame
df_filtered = neurons_df.drop(first_occurrences)

# Extract and adjust indices based on 'frame_number' from index_df
indices_black = df_filtered['frame_number'].values - 1114

# Convert indices to integers
indices_black = indices_black.astype(int)

indices_black

In [ ]:
# Animate function
def animate_trajectories(data, highlight_indices_red, highlight_indices_black, start_frame=0, end_frame=None, output_file=CONFIG['animations_path']):
    if end_frame is None or end_frame > len(data):
        end_frame = len(data)

    # Smooth the data
    smoothed_data = smooth_trajectory(data[start_frame:end_frame])

    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    ax.set_xlim(np.min(smoothed_data[:, 0]), np.max(smoothed_data[:, 0]))
    ax.set_ylim(np.min(smoothed_data[:, 1]), np.max(smoothed_data[:, 1]))
    ax.set_zlim(np.min(smoothed_data[:, 2]), np.max(smoothed_data[:, 2]))

    # Parameters for animation
    sweep_duration = 15
    hold_duration = 10
    total_duration = sweep_duration + hold_duration
    fps = 30

    # Initialize the line
    line, = ax.plot([], [], [], lw=2, color='blue')

    # Setup the writer
    writer = animation.FFMpegWriter(fps=fps, bitrate=500)
    
    with writer.saving(fig, output_file, 100):
        pON_red = False
        pON_black = False
        count_red = 0
        count_black = 0
        pcount = -1
        for i in range(start_frame, start_frame + 00):
            ax.cla()  # Clear the axes
            ax.set_xlim(np.min(smoothed_data[:, 0]), np.max(smoothed_data[:, 0]))
            ax.set_ylim(np.min(smoothed_data[:, 1]), np.max(smoothed_data[:, 1]))
            ax.set_zlim(np.min(smoothed_data[:, 2]), np.max(smoothed_data[:, 2]))
            
            # Add axis labels
            ax.set_xlabel('X axis')
            ax.set_ylabel('Y axis')
            ax.set_zlabel('Z axis')

            ax.plot(smoothed_data[start_frame:, 0], smoothed_data[start_frame:, 1], smoothed_data[start_frame:, 2], lw=1, color='blue', alpha=0.1, zorder=5)
            ax.plot(smoothed_data[i-20:i, 0], smoothed_data[i-20:i, 1], smoothed_data[i-20:i, 2], lw=2, color='orange', zorder=10)
            
            if (i in highlight_indices_red) or pON_red:
                ax.plot(smoothed_data[i-count_red:i, 0], smoothed_data[i-count_red:i, 1], smoothed_data[i-count_red:i, 2], lw=3, color='red', zorder=10)
                pON_red = True
                count_red += 1
                if count_red > 6:
                    count_red = 0
                    pON_red = False
                    pcount += 1
            
            if (i in highlight_indices_black) or pON_black:
                ax.plot(smoothed_data[i-count_black:i, 0], smoothed_data[i-count_black:i, 1], smoothed_data[i-count_black:i, 2], lw=3, color='black', zorder=10)
                pON_black = True
                count_black += 1
                if count_black > 6:
                    count_black = 0
                    pON_black = False
                    pcount += 1

            writer.grab_frame()

In [ ]:
output_file = 'trajectory_animation4.mp4'
animate_trajectories(small_C, indices , indices_black, start_frame=0, end_frame=1000, output_file=output_file)

## Final trajectory

In [ ]:
output_file = 'whole_trajectory_animation.mp4'
animate_trajectories(small_C, indices , indices_black, start_frame=0, end_frame=55000, output_file=output_file)